> 在线讲义：[第 02 章 · 一次生成请求的处理过程：权重加载一次，每个 token 都要再读一遍](https://wikieden.github.io/ai-infra-from-constraints/lessons/02-request-lifecycle.html) · 本书《从约束推导 AI Infra》

# 第 2 章 · 一次生成请求的处理过程：权重加载一次，每个 token 都要再读一遍

配套讲义：仓库里 `ai-infra/notes/lessons/02-request-lifecycle.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 路由选更闲的实例 | 选排队最少的 | `route(loads)` | `{"A": 6, "B": 2}` → `B` |
| 第三步 4 个输入、3 个输出 | $N$ 次前向，KV $= L_{\text{in}} + N - 1$ | `simulate(4, 3)` | 3 次前向，KV 4 → 5 → 6 |
| 第四步 Qwen3-8B 2048 输入、50 输出 | $N \times W_{\text{step}}$ | `weight_read_gb(50)` | 约 757 GB |
| 第四步 2048 个 token 的 KV | 份数 × 144 KiB | `kv_mib(2048)` | 288 MiB |
| 第四步 请求文字和读权重差多少倍 | 757 GB ÷ 8 KB | `ratio()` | 约 $9.5 \times 10^7$ 倍 |

Qwen3-8B 的 16.38 GB、15.14 GB、144 KiB 取自 AII 第 1 章"本章数字速查"；实例负载、4 个输入 token、请求文字 8 KB 是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
WEIGHT_LOAD_GB = 16.38  # 完整 BF16 权重：加载进显存的量
WEIGHT_STEP_GB = 15.14  # 一次前向的主要权重读取（嵌入层只读当前 token 那一行）
KV_KIB_PER_TOKEN = 144  # 每个上下文 token 的全模型 KV

**函数 `route`**：路由器：在各实例里挑排队请求最少的那个（并列时取第一个）。

In [ ]:
def route(loads: dict[str, int]) -> str:
    """路由器：在各实例里挑排队请求最少的那个（并列时取第一个）。"""
    return min(loads, key=loads.get)

**函数 `simulate`**：逐次前向：返回 (阶段, 送进几个 token, 读几份旧 KV, 结束时 KV 份数)。

In [ ]:
def simulate(prompt_len: int, out_len: int) -> list[tuple[str, int, int, int]]:
    """逐次前向：返回 (阶段, 送进几个 token, 读几份旧 KV, 结束时 KV 份数)。"""
    kv, steps = 0, []
    for i in range(out_len):  # 输出 N 个 token = N 次前向
        fed = prompt_len if i == 0 else 1  # prefill 送进全部输入，decode 只送 1 个
        read_old = kv  # 先读已有的整份 KV
        kv += fed  # 再追加这次送进来的 token 的 k、v
        steps.append(("prefill" if i == 0 else "decode", fed, read_old, kv))
    return steps

**函数 `weight_read_gb`**：权重读取总量 ≈ N × W_step（每次前向读一遍）。

In [ ]:
def weight_read_gb(out_len: int, step_gb: float = WEIGHT_STEP_GB) -> float:
    """权重读取总量 ≈ N × W_step（每次前向读一遍）。"""
    return out_len * step_gb

**函数 `kv_mib`**：KV 缓存大小（MiB）= 份数 × 每份 KiB ÷ 1024。

In [ ]:
def kv_mib(tokens: int, kib_per_token: float = KV_KIB_PER_TOKEN) -> float:
    """KV 缓存大小（MiB）= 份数 × 每份 KiB ÷ 1024。"""
    return tokens * kib_per_token / 1024

**函数 `ratio`**：内部读取量是请求文字的多少倍：先都换成字节（GB = 1e9，KB = 1e3）。

In [ ]:
def ratio(read_gb: float = 757.0, request_kb: float = 8.0) -> float:
    """内部读取量是请求文字的多少倍：先都换成字节（GB = 1e9，KB = 1e3）。"""
    return read_gb * 1e9 / (request_kb * 1e3)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：路由器挑更闲的实例（编的负载）")
loads = {"A": 6, "B": 2}
print(f"  各实例排队数 {loads} → 派给 {route(loads)}")

In [ ]:
print("\n第三步：输入 4 个 token，输出 3 个 token（编的）")
for n, (phase, fed, read_old, kv) in enumerate(simulate(4, 3), 1):
    print(f"  第 {n} 次前向 {phase:7s} 送进 {fed} 个 token，读 {read_old} 份旧 KV，结束时 KV {kv} 份")
print("  输出 N 个 token = N 次前向；结束时 KV = 输入长度 + N − 1 = 4 + 2 = 6")

In [ ]:
print("\n第四步：Qwen3-8B、BF16，2048 个 token 的上下文，输出 50 个 token")
print(f"  权重从存储加载 1 次：{WEIGHT_LOAD_GB} GB（实例启动时）")
print(f"  权重从显存读 50 次：50 × {WEIGHT_STEP_GB} ≈ {weight_read_gb(50):.0f} GB")
print(f"  prefill 结束时 KV：2048 × {KV_KIB_PER_TOKEN} KiB = {kv_mib(2048):.0f} MiB；之后每步追加 {KV_KIB_PER_TOKEN} KiB")
print(f"  请求文字约 8 KB（编的）→ 读权重总量是它的 {ratio():.2e} 倍")

## 3. 代码里才看得到的

1. 循环跑 $N$ 次，但 KV 只追加到 $L_{\text{in}} + N - 1$：最后一个输出 token 没被送进模型，它的 k、v 从没算过。要接着对话时，它会在下一次请求里补算。
2. 权重加载量（16.38 GB）和每步读取量（15.14 GB）不是同一个数，两个常量要分开写，混用会多算约 8%。
3. 原书的 GB 是 $10^9$ 字节，KV 的 KiB / MiB 是 $2^{10}$ / $2^{20}$；`ratio()` 里先把 8 KB 换成字节再除。

## 4. 自己改一改

- 把 `simulate(4, 3)` 的输入长度改成 1、改成 10，看 decode 步读的旧 KV 份数怎么变。
- 把 `route` 的负载改成两个实例一样忙，看它怎么挑（提示：Python 的 `min` 遇到并列取第一个）。
- 把 `weight_read_gb` 的输出 token 数从 50 改成 5，看读取总量怎么变，再想想"加载 1 次"变不变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 路由选店，调度排单；
> 读题一口气，写字一个个；
> 菜谱搬一次，每锅翻一遍；
> 柜内 scale-up，柜外 scale-out。